# E2E Quality-Loss Smoke Test (Colab)

This notebook is the setup smoke test for the full E2E quality-loss training run. It is run after the fresh CNN warm-start checkpoint is produced.

It runs a small two-epoch E2E smoke test with:
- ISP unfrozen;
- pretrained CNN loaded from `cnn_pretrained.pth`;
- separate optimizer groups: ISP `1e-4`, CNN `5e-4`;
- quality-aligned loss (the same surrogate validated by the quality-overfit sanity);
- best checkpoint selected by normalized `VIF_norm + a*NRQM_norm + b*UNIQUE_norm`.

The train HDF5 is intentionally tiny. This is a setup smoke test, not the full E2E training run. Composite values are logged and used for best-checkpoint selection, but epoch-to-epoch monotonic growth is treated as a diagnostic because the smoke validation uses only a tiny number of frames.

## Required Google Drive files

Create this folder:

`MyDrive/ISP_colab/e2e_smoke/`

Upload these files:

- `e2e_smoke_bundle.zip`
- `e2e_smoke_train.h5`
- `cnn_pretrained.pth` from the fresh CNN warm-start
- `day_val_mini.bin`
- `day_val_mini.yuv`
- `night_val_mini.bin`
- `night_val_mini.yuv`

Outputs are written to:

`MyDrive/ISP_colab/e2e_smoke_outputs/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Configuration

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ISP_colab')
DRIVE_INPUT = DRIVE_ROOT / 'e2e_smoke'
DRIVE_OUTPUT = DRIVE_ROOT / 'e2e_smoke_outputs'

LOCAL_REPO = Path('/content/ISP')
LOCAL_TRAIN_H5 = LOCAL_REPO / 'dataset' / 'e2e_smoke_train.h5'
LOCAL_CKPT = LOCAL_REPO / 'artifacts' / 'checkpoints' / 'cnn_pretrained.pth'

CLEAR_OLD_OUTPUTS = True

EPOCHS = 2
BATCH_SIZE = 1
LR_ISP = 1e-4
LR_CNN = 5e-4
LAMBDA_UV = 1.0

LOSS = 'quality'
BEST_CRITERION = 'composite'
W_SSIM = 1.0
W_VIF = 0.3
W_UNIQUE = 0.1
W_UV = 1.0

EVAL_EVERY = 1
EVAL_MAX_FRAMES = 1
CHECKPOINT_EVERY = 2
MAX_TRAIN_BATCHES = 1
SEED = 42
NUM_WORKERS = 0

print(f'Drive input:  {DRIVE_INPUT}')
print(f'Drive output: {DRIVE_OUTPUT}')
print(f'Local repo:   {LOCAL_REPO}')

## 2. Check GPU

In [ ]:
import subprocess
import torch

try:
    print(subprocess.check_output([
        'nvidia-smi',
        '--query-gpu=name,memory.total,driver_version',
        '--format=csv'
    ]).decode())
except Exception as exc:
    print(f'nvidia-smi is not available: {exc}')

print(f'torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('Please switch Colab runtime to GPU before running the E2E smoke.')
print(f'device: {torch.cuda.get_device_name(0)}')

## 3. Install dependencies

In [ ]:
!pip install -q pyiqa tomli h5py tqdm pillow

## 4. Prepare local workspace

In [ ]:
import os
import shutil
import time
import zipfile

def required(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return path

code_zip = required(DRIVE_INPUT / 'e2e_smoke_bundle.zip')
train_h5 = required(DRIVE_INPUT / 'e2e_smoke_train.h5')
ckpt = required(DRIVE_INPUT / 'cnn_pretrained.pth')
mini_files = {
    name: required(DRIVE_INPUT / name)
    for name in ['day_val_mini.bin', 'day_val_mini.yuv', 'night_val_mini.bin', 'night_val_mini.yuv']
}
if LOCAL_REPO.exists():
    print(f'Removing old local workspace: {LOCAL_REPO}')
    shutil.rmtree(LOCAL_REPO)
LOCAL_REPO.mkdir(parents=True, exist_ok=True)

if CLEAR_OLD_OUTPUTS and DRIVE_OUTPUT.exists():
    assert DRIVE_OUTPUT.name == 'e2e_smoke_outputs'
    assert str(DRIVE_OUTPUT).startswith(str(DRIVE_ROOT))
    print(f'Removing old E2E smoke outputs: {DRIVE_OUTPUT}')
    shutil.rmtree(DRIVE_OUTPUT)
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

print(f'Unzipping {code_zip.name}...', end=' ', flush=True)
t0 = time.time()
with zipfile.ZipFile(code_zip, 'r') as zf:
    zf.extractall(LOCAL_REPO)
print(f'done ({time.time() - t0:.1f}s)')

LOCAL_TRAIN_H5.parent.mkdir(parents=True, exist_ok=True)
print(f'Copying train HDF5...', end=' ', flush=True)
t0 = time.time()
shutil.copy2(train_h5, LOCAL_TRAIN_H5)
print(f'done ({LOCAL_TRAIN_H5.stat().st_size / 1024**2:.2f} MiB, {time.time() - t0:.1f}s)')

LOCAL_CKPT.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(ckpt, LOCAL_CKPT)
print(f'Copied checkpoint: {LOCAL_CKPT}')

(LOCAL_REPO / 'data').mkdir(parents=True, exist_ok=True)
for name, src in mini_files.items():
    dst = LOCAL_REPO / 'data' / name
    shutil.copy2(src, dst)
    print(f'Copied {name}: {dst.stat().st_size / 1024**2:.1f} MiB')

checks = [
    LOCAL_REPO / 'scripts' / 'run_e2e_train.py',
    LOCAL_REPO / 'isp' / 'training' / 'training_utils.py',
    LOCAL_REPO / 'isp' / 'training' / 'quality_loss.py',
    LOCAL_REPO / 'metrics' / 'vif.py',
    LOCAL_REPO / 'data' / 'imx623.toml',
    LOCAL_REPO / 'dataset' / 'splits_mini.json',
    LOCAL_REPO / 'artifacts' / 'baselines' / 'norm_weights.json',
    LOCAL_TRAIN_H5,
    LOCAL_CKPT,
]
for path in checks:
    print(f'{path.relative_to(LOCAL_REPO)}: {"OK" if path.exists() else "MISSING"}')
    if not path.exists():
        raise FileNotFoundError(path)

## 5. Run 2-epoch E2E smoke

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, '-B', 'scripts/run_e2e_train.py',
    '--device', 'cuda',
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--lr-isp', str(LR_ISP),
    '--lr-cnn', str(LR_CNN),
    '--lambda-uv', str(LAMBDA_UV),
    '--loss', LOSS,
    '--best-criterion', BEST_CRITERION,
    '--w-ssim', str(W_SSIM),
    '--w-vif', str(W_VIF),
    '--w-unique', str(W_UNIQUE),
    '--w-uv', str(W_UV),
    '--eval-every', str(EVAL_EVERY),
    '--eval-max-frames', str(EVAL_MAX_FRAMES),
    '--checkpoint-every', str(CHECKPOINT_EVERY),
    '--max-train-batches', str(MAX_TRAIN_BATCHES),
    '--seed', str(SEED),
    '--num-workers', str(NUM_WORKERS),
    '--train-h5', 'dataset/e2e_smoke_train.h5',
    '--splits-json', 'dataset/splits_mini.json',
    '--ckpt', 'artifacts/checkpoints/cnn_pretrained.pth',
    '--norm-weights', 'artifacts/baselines/norm_weights.json',
    '--output-dir', str(DRIVE_OUTPUT),
    '--history-name', 'smoke_history.json',
    '--ckpt-prefix', 'smoke',
]

print('Running:')
print(' '.join(cmd))
print()

proc = subprocess.Popen(cmd, cwd=str(LOCAL_REPO), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end='')
    ret = proc.wait()
finally:
    if proc.poll() is None:
        proc.terminate()

print(f'\n[subprocess exit={ret}]')
if ret != 0:
    raise RuntimeError(f'E2E smoke failed with exit code {ret}')

## 6. Verify smoke checks

In [ ]:
import json
import math

history_path = DRIVE_OUTPUT / 'smoke_history.json'
best_path = DRIVE_OUTPUT / 'smoke_best.pth'
resume_path = DRIVE_OUTPUT / 'smoke_resume.pth'
with open(history_path, 'r') as f:
    history = json.load(f)

epochs = [row for row in history if row.get('epoch', 0) > 0]
if len(epochs) != 2:
    raise AssertionError(f'Expected 2 smoke epochs, got {len(epochs)}')

def grad_ok(row):
    grads = row['grad_means']
    return all(grads.get(name) is not None and grads.get(name) > 0.0 for name in [
        'ccm_grad_mean', 'gamma_grad_mean',
        'cnn_head_grad_mean', 'cnn_body_grad_mean', 'cnn_tail_grad_mean',
    ])

start = epochs[0]
final = epochs[-1]
night_start = start['val_per_scene']['night']['vif']
night_final = final['val_per_scene']['night']['vif']
night_ok = night_final >= 0.9 * night_start
composites = [row.get('val_composite') for row in epochs]
finite_composites = all(v is not None and math.isfinite(float(v)) for v in composites)
best_composite = max(float(v) for v in composites) if finite_composites else None
final_composite_delta = float(final['val_composite']) - float(start['val_composite']) if finite_composites else None

print('E2E smoke summary:')
for row in epochs:
    night = row['val_per_scene']['night']['vif']
    print(f"  epoch {row['epoch']}: loss={row['train_loss']:.6f}, composite={row['val_composite']:.4f}, night_vif={night:.4f}, grads_ok={grad_ok(row)}")
if finite_composites:
    print(f'  best composite in smoke: {best_composite:.4f}')
    print(f'  final - first composite: {final_composite_delta:+.4f} (diagnostic only)')

checks = {
    'epoch_count_is_2': len(epochs) == 2,
    'all_grads_present': all(grad_ok(row) for row in epochs),
    'isp_params_changed': any(row['isp_params'] != history[0]['isp_params'] for row in epochs),
    'finite_composite_values': finite_composites,
    'best_checkpoint_saved': best_path.exists(),
    'resume_checkpoint_saved': resume_path.exists(),
    'night_vif_not_collapsed': night_ok,
}

print('\nChecks:')
for name, ok in checks.items():
    print(f'  {name:<26s}: {"OK" if ok else "FAIL"}')

if not all(checks.values()):
    raise AssertionError('E2E smoke did not pass all checks.')

print(f'\nOutputs saved to: {DRIVE_OUTPUT}')